# 3. Native Windows GPU smoke test and fine-tuning
Install the separate Windows environment from the README. Run the synthetic smoke test **before** annotation proposals or long training. It verifies real SAM 3 mask loss, gradients, an optimizer step, prediction, strict checkpoint reload, and peak VRAM; synthetic outputs are not scientific results.

Training uses the pinned official SAM 3 model, COCO loader, matcher, collator, transforms, and segmentation losses. A notebook-owned single-process loop uses `normalization="local"` instead of distributed collectives. Negative class queries are retained. No Slurm, NCCL, process spawning, or model compilation is required. Triton is still required by upstream loss kernels; native Windows has not been exercised on this CPU machine.

Accuracy selection uses validation mask AP only. Test images are never loaded here. Frozen encoders provide a memory baseline; progressively larger unfreezing profiles are attempted by the smoke test. Profiles that exceed VRAM are recorded and skipped, not silently trained at lower resolution.


In [ ]:
from pathlib import Path
import json
import numpy as np
from PIL import Image
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataset_notebooks').is_dir() and (p / 'notebooks/reproduction.py').is_file())
sys.path.insert(0, str(ROOT / 'dataset_notebooks'))
from runtime import fresh_run, write_json, read_json, sha256, environment, load_model, SAM3_REVISION


In [ ]:
RUN_SMOKE_TEST = False
RUN_TRAINING = False
ALLOW_WEIGHT_DOWNLOAD = False
PRETRAINED_CHECKPOINT = None
DATA = ROOT/'experiments/reproductions/sam3-reviewed-export-v1'
RUN_NAME = 'sam3-training-v1'
SMOKE_RUN_NAME = 'sam3-smoke-v1'
SMOKE_REPORT = ROOT/'experiments/reproductions'/SMOKE_RUN_NAME/'smoke.json'
RESUME_CHECKPOINT = None  # Resume a single interrupted trial into a fresh run.
OPTIONS = {'resolution':1008,'batch_size':1,'accumulation':8,'epochs':60,'patience':10,
           'learning_rates':[1e-5,3e-5],'seeds':[17,42,2026],
           'profiles':['heads','last4','last8','full'],'weight_decay':0.01,
           'clip_norm':0.1,'augmentation':True}


## Official data and mask-loss adapters

Keep negative class queries. Decode instance masks before geometric augmentation, then resize to SAM 3’s 1008-pixel input. Use explicit instance focal/Dice losses; the upstream example otherwise defaults to box-only training.


In [ ]:
def build_data(folder,training,options):
    from functools import partial
    from sam3.train.data.sam3_image_dataset import Sam3ImageDataset
    from sam3.train.data.coco_json_loaders import COCO_FROM_JSON
    from sam3.train.transforms import basic_for_api as T
    from sam3.train.transforms.segmentation import DecodeRle
    transforms=[DecodeRle()]
    if training and options['augmentation']:
        transforms += [T.RandomHorizontalFlip(consistent_transform=True,p=.5),
            T.ColorJitter(consistent_transform=True,brightness=.15,contrast=.15,saturation=.15,hue=.025),
            T.RandomAffine(degrees=15,consistent_transform=True,scale=(.9,1.1),translate=(.05,.05))]
    transforms += [T.RandomResizeAPI(sizes=options['resolution'],consistent_transform=True,square=True),
                   T.ToTensorAPI(),T.NormalizeAPI(mean=[.5]*3,std=[.5]*3)]
    dataset = Sam3ImageDataset(img_folder=str(folder),ann_file=str(Path(folder)/'annotations.json'),
        transforms=[T.ComposeAPI(transforms)],max_ann_per_img=200,multiplier=1,
        training=training,load_segmentation=True,use_caching=False,
        coco_json_loader=partial(COCO_FROM_JSON,include_negatives=True,category_chunk_size=1))
    dataset._MAX_RETRIES=1  # Do not silently replace malformed/overcrowded images.
    return dataset


def build_criterion():
    import torch
    from sam3.train.matcher import BinaryHungarianMatcherV2, BinaryOneToManyMatcher
    from sam3.train.loss.loss_fns import Boxes, IABCEMdetr, Masks
    from sam3.train.loss.sam3_loss import Sam3LossWrapper
    return Sam3LossWrapper(normalization='local',
        matcher=BinaryHungarianMatcherV2(focal=True,cost_class=2,cost_bbox=5,cost_giou=2,alpha=.25,gamma=2,stable=False),
        o2m_matcher=BinaryOneToManyMatcher(alpha=.3,threshold=.4,topk=4),o2m_weight=2,
        use_o2m_matcher_on_o2m_aux=False,scale_by_find_batch_size=True,
        loss_fns_find=torch.nn.ModuleList([
            Boxes(weight_dict={'loss_bbox':5.,'loss_giou':2.}),
            IABCEMdetr(weak_loss=False,weight_dict={'loss_ce':20.,'presence_loss':20.},
                pos_weight=10,alpha=.25,gamma=2,use_presence=True,pos_focal=False,pad_n_queries=200),
            Masks(weight_dict={'loss_mask':200.,'loss_dice':10.},compute_aux=False,focal_alpha=.25,focal_gamma=2)]))


## Memory profiles and optimization

The text encoder remains frozen. Profiles progressively unfreeze the vision encoder while retaining pretrained weights. Head and backbone learning rates differ by a factor of ten.


In [ ]:
def set_trainable(model,profile):
    if profile not in ('heads','last4','last8','full'):raise ValueError(profile)
    for p in model.parameters():p.requires_grad_(True)
    for p in model.backbone.language_backbone.parameters():p.requires_grad_(False)
    vision=model.backbone.vision_backbone
    if profile!='full':
        for p in vision.parameters():p.requires_grad_(False)
        n={'heads':0,'last4':4,'last8':8}[profile]
        if n:
            for block in vision.trunk.blocks[-n:]:
                for p in block.parameters():p.requires_grad_(True)
    vision.trunk.use_act_checkpoint=True
    model.use_act_checkpoint_seg_head=True
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_optimizer(model,lr,options):
    import torch
    backbone=[];heads=[]
    for name,p in model.named_parameters():
        if p.requires_grad:(backbone if name.startswith('backbone.') else heads).append(p)
    groups=[{'params':heads,'lr':lr}]
    if backbone:groups.append({'params':backbone,'lr':lr*.1})
    return torch.optim.AdamW(groups,weight_decay=options['weight_decay'])


## Training loss and validation selection

Use SAM 3’s own batch representation, matching, and loss modules. Validation runs image/text inference without annotation prompts and selects checkpoints by COCO mask AP.


In [ ]:
def forward_loss(model,criterion,datapoints):
    from sam3.train.data.collator import collate_fn_api
    from sam3.model.utils.misc import copy_data_to_device
    batch=collate_fn_api(datapoints,dict_key='all',with_seg_masks=True)['all']
    batch=copy_data_to_device(batch,'cuda')
    stages=model(batch)
    losses=criterion(stages,[model.back_convert(t) for t in batch.find_targets])
    if not any('loss_mask' in k for k in losses):raise RuntimeError('Mask loss is missing.')
    return losses['core_loss'],losses


def validation_ap(model,folder,resolution):
    import torch
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    from pycocotools import mask as mask_api
    from sam3.model.sam3_image_processor import Sam3Processor
    folder=Path(folder);gt=COCO(str(folder/'annotations.json'));predictions=[]
    model.eval();processor=Sam3Processor(model,resolution=resolution,confidence_threshold=.01)
    with torch.inference_mode(),torch.autocast('cuda',dtype=torch.bfloat16):
        for info in gt.dataset['images']:
            state=processor.set_image(Image.open(folder/info['file_name']).convert('RGB'))
            for category,prompt in [(1,'tobacco plant'),(2,'weed plant')]:
                processor.reset_all_prompts(state)
                output=processor.set_text_prompt(prompt=prompt,state=state)
                for m,s in zip(output['masks'],output['scores']):
                    m=m.squeeze(0).cpu().numpy().astype('uint8')
                    if not m.any():continue
                    rle=mask_api.encode(np.asfortranarray(m));rle['counts']=rle['counts'].decode('ascii')
                    predictions.append({'image_id':info['id'],'category_id':category,'segmentation':rle,'score':float(s)})
    if predictions:dt=gt.loadRes(predictions)
    else:
        dt=COCO();dt.dataset={'images':gt.dataset['images'],'categories':gt.dataset['categories'],'annotations':[]};dt.createIndex()
    ev=COCOeval(gt,dt,'segm');ev.evaluate();ev.accumulate();ev.summarize()
    metrics={'mask_AP':float(ev.stats[0]),'mask_AP50':float(ev.stats[1])}
    for idx,category in enumerate(ev.params.catIds):
        precision=ev.eval['precision'][:,:,idx,0,-1];valid=precision[precision>=0]
        metrics[f'class_{category}_AP']=float(valid.mean()) if len(valid) else None
    return metrics


## Resumable checkpoints

Save the full model, optimizer, RNG states, data hashes, and configuration atomically. These checkpoints use an explicit format marker and strict loading.


In [ ]:
def checkpoint_payload(model,optimizer,epoch,best,stale,trial,data_hashes):
    import torch,random
    return {'format':'tobacco-sam3-v1','sam3_revision':SAM3_REVISION,
        'model':model.state_dict(),'optimizer':optimizer.state_dict(),'epoch':epoch,'best_AP':best,
        'stale':stale,'trial':trial,'data_hashes':data_hashes,
        'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
        'initial_checkpoint_sha256':model._source_checkpoint_sha256,
        'python_rng':random.getstate(),'numpy_rng':{
            'name':np.random.get_state()[0],'keys':np.random.get_state()[1].tolist(),
            'pos':np.random.get_state()[2],'has_gauss':np.random.get_state()[3],'cached_gauss':np.random.get_state()[4]}}


def save_checkpoint(path,payload):
    import torch
    path=Path(path);temporary=path.with_suffix('.tmp')
    torch.save(payload,temporary);temporary.replace(path)


## A complete training trial

Field-balanced sampling prevents the two large campaigns from dominating solely by patch count. Accumulation handles the final partial group correctly. Epoch-end validation determines best.pt and early stopping; failures are recorded and raised.


In [ ]:
def train_trial(data,folder,trial,checkpoint,allow_download,resume=None):
    import torch,random,time,math
    if trial['batch_size']!=1:raise ValueError('This memory profile supports microbatch 1; change accumulation instead.')
    from collections import Counter
    seed=trial['seed'];random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    model=load_model(checkpoint,allow_download,training=True);set_trainable(model,trial['profile'])
    criterion=build_criterion().cuda();optimizer=make_optimizer(model,trial['lr'],trial)
    ds=build_data(Path(data)/'train',True,trial)
    annotations=read_json(Path(data)/'train/annotations.json')
    # COCO loader sorts image IDs and emits two class queries per image.
    infos=sorted(annotations['images'],key=lambda x:x['id']);counts=Counter(x['campaign'] for x in infos)
    weights=torch.tensor([1/counts[x['campaign']] for x in infos for _ in range(2)],dtype=torch.double)
    if len(weights)!=len(ds):raise ValueError('Unexpected SAM query enumeration.')
    hashes={s:sha256(Path(data)/s/'annotations.json') for s in ('train','val')}
    start=0;best=-1.;stale=0
    if resume:
        p=torch.load(resume,map_location='cpu',weights_only=True)
        if p['trial']!=trial or p['data_hashes']!=hashes:raise ValueError('Resume config or annotations differ.')
        model.load_state_dict(p['model'],strict=True);optimizer.load_state_dict(p['optimizer'])
        start=p['epoch']+1;best=p['best_AP'];stale=p['stale']
        torch.set_rng_state(p['torch_rng']);torch.cuda.set_rng_state_all(p['cuda_rng']);random.setstate(p['python_rng'])
        n=p['numpy_rng'];np.random.set_state((n['name'],np.array(n['keys'],dtype='uint32'),n['pos'],n['has_gauss'],n['cached_gauss']))
        del p
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=False)
    if resume:
        import shutil
        previous_best=Path(resume).parent/'best.pt'
        if not previous_best.is_file():raise FileNotFoundError('Resume requires sibling best.pt to preserve prior selection.')
        shutil.copy2(previous_best,folder/'best.pt')
    write_json(folder/'trial.json',trial);history=[]
    try:
        for epoch in range(start,trial['epochs']):
            model.train();model.backbone.language_backbone.eval()
            if trial['profile']=='heads':model.backbone.vision_backbone.eval()
            ds.set_epoch(epoch)
            order=torch.multinomial(weights,len(ds),replacement=True).tolist()
            warm=min(1.,(epoch+1)/3);cos=.5*(1+math.cos(math.pi*epoch/trial['epochs']))
            for group in optimizer.param_groups:
                base=trial['lr']*(.1 if group is not optimizer.param_groups[0] else 1.)
                group['lr']=base*warm*cos
            optimizer.zero_grad(set_to_none=True);total=0.;started=time.perf_counter()
            for step,index in enumerate(order):
                group_start=step//trial['accumulation']*trial['accumulation']
                divisor=min(trial['accumulation'],len(order)-group_start)
                with torch.autocast('cuda',dtype=torch.bfloat16):loss,parts=forward_loss(model,criterion,[ds[index]])
                if not torch.isfinite(loss):raise FloatingPointError('Non-finite training loss')
                total+=float(loss.detach());(loss/divisor).backward()
                if (step+1)%trial['accumulation']==0 or step+1==len(order):
                    torch.nn.utils.clip_grad_norm_(model.parameters(),trial['clip_norm'],error_if_nonfinite=True)
                    optimizer.step();optimizer.zero_grad(set_to_none=True)
            metrics=validation_ap(model,Path(data)/'val',trial['resolution'])
            improved=metrics['mask_AP']>best
            best=max(best,metrics['mask_AP']);stale=0 if improved else stale+1
            history.append({'epoch':epoch,'loss':total/len(order),'seconds':time.perf_counter()-started,**metrics})
            payload=checkpoint_payload(model,optimizer,epoch,best,stale,trial,hashes)
            save_checkpoint(folder/'last.pt',payload)
            if improved:save_checkpoint(folder/'best.pt',payload)
            write_json(folder/'history.json',history);print(history[-1])
            if stale>=trial['patience']:break
        write_json(folder/'status.json',{'status':'complete','best_AP':best})
    except Exception as e:
        write_json(folder/'status.json',{'status':'failed','error':repr(e)});raise
    finally:
        del model,criterion,optimizer;import gc;gc.collect();torch.cuda.empty_cache()
    return {'folder':str(folder),'best_AP':best,'checkpoint':str(folder/'best.pt')}


## Hardware smoke test

Exercise both a positive and an absent-class query on synthetic data. Measure CUDA time/VRAM, verify mask loss and finite gradients, and save/reload weights before accepting a memory profile. This is software validation, not field performance.


In [ ]:
def synthetic_smoke(folder,checkpoint,allow_download,options):
    import torch,time,tempfile,gc
    from pycocotools import mask as mask_api
    folder=Path(folder);reports=[]
    with tempfile.TemporaryDirectory() as temporary:
        temp=Path(temporary);rgb=np.zeros((352,480,3),np.uint8);rgb[:]=(110,90,60)
        rgb[80:230,90:240]=(30,180,50);Image.fromarray(rgb).save(temp/'fixture.png')
        mask=np.zeros((352,480),np.uint8);mask[80:230,90:240]=1
        rle=mask_api.encode(np.asfortranarray(mask));rle['counts']=rle['counts'].decode('ascii')
        write_json(temp/'annotations.json',{'info':{},'images':[{'id':1,'file_name':'fixture.png','width':480,'height':352}],
            'categories':[{'id':1,'name':'tobacco plant'},{'id':2,'name':'weed plant'}],
            'annotations':[{'id':1,'image_id':1,'category_id':1,'bbox':[90,80,150,150],'area':22500,'iscrowd':0,'segmentation':rle}]})
        for profile in options['profiles']:
            model=criterion=optimizer=None
            try:
                model=load_model(checkpoint,allow_download,training=True)
                count=set_trainable(model,profile);criterion=build_criterion().cuda();optimizer=make_optimizer(model,1e-5,options)
                ds=build_data(temp,False,options);torch.cuda.reset_peak_memory_stats();torch.cuda.synchronize();t=time.perf_counter()
                losses=[]
                # Positive and empty class queries must both backpropagate.
                for index in (0,1):
                    optimizer.zero_grad(set_to_none=True)
                    with torch.autocast('cuda',dtype=torch.bfloat16):loss,parts=forward_loss(model,criterion,[ds[index]])
                    if not torch.isfinite(loss):raise FloatingPointError('Non-finite smoke loss')
                    loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),.1,error_if_nonfinite=True);optimizer.step()
                    losses.append(float(loss.detach()))
                torch.cuda.synchronize();elapsed=time.perf_counter()-t;peak=torch.cuda.max_memory_allocated()/2**30
                del loss,parts
                optimizer.zero_grad(set_to_none=True)
                metrics=validation_ap(model,temp,options['resolution'])
                p=folder/f'smoke-{profile}.pt'
                save_checkpoint(p,{'format':'tobacco-sam3-v1','sam3_revision':SAM3_REVISION,'model':model.state_dict()})
                del optimizer,criterion,model;optimizer=criterion=model=None;gc.collect();torch.cuda.empty_cache()
                reloaded=load_model(p,False);metrics2=validation_ap(reloaded,temp,options['resolution'])
                if metrics!=metrics2:raise AssertionError('Reloaded predictions changed metrics.')
                del reloaded;gc.collect();torch.cuda.empty_cache()
                reports.append({'profile':profile,'status':'passed','trainable_parameters':count,
                    'peak_GiB':peak,'seconds_per_query':elapsed/2,'losses':losses,'reload_verified':True})
            except torch.cuda.OutOfMemoryError as e:
                reports.append({'profile':profile,'status':'out_of_memory','error':str(e)})
            finally:
                del model,criterion,optimizer;gc.collect();torch.cuda.empty_cache()
            write_json(folder/'smoke.json',{'sam3_revision':SAM3_REVISION,'resolution':options['resolution'],
                'environment':environment(),'profiles':reports})
    return reports


In [ ]:
if RUN_SMOKE_TEST:
    smoke=fresh_run(ROOT,SMOKE_RUN_NAME)
    print(synthetic_smoke(smoke,PRETRAINED_CHECKPOINT,ALLOW_WEIGHT_DOWNLOAD,OPTIONS))
if RUN_TRAINING:
    import itertools
    smoke=read_json(SMOKE_REPORT)
    if smoke['sam3_revision']!=SAM3_REVISION or smoke['resolution']!=OPTIONS['resolution']:
        raise ValueError('Smoke configuration differs; rerun smoke test.')
    import torch
    current=environment()
    if any(current.get(k)!=smoke['environment'].get(k) for k in ('torch','cuda','gpu')):
        raise ValueError('GPU/PyTorch environment changed; rerun smoke test.')
    profiles=[r['profile'] for r in smoke['profiles'] if r['status']=='passed' and r['profile'] in OPTIONS['profiles']]
    if not profiles:raise RuntimeError('No profile passed the native GPU smoke test.')
    for split in ('train','val'):
        coco=read_json(DATA/split/'annotations.json')
        for info in coco['images']:
            if sha256(DATA/split/info['file_name'])!=info['image_sha256']:raise ValueError('Exported image changed.')
            if not info['annotation_provenance']['approved']:raise ValueError('Unapproved annotations.')
        from collections import Counter
        counts=Counter((a['image_id'],a['category_id']) for a in coco['annotations'])
        if max(counts.values(),default=0)>200:raise ValueError('More than 200 instances per class/image: create smaller reviewed training crops before proceeding.')
    run=fresh_run(ROOT,RUN_NAME);write_json(run/'environment.json',environment());write_json(run/'options.json',OPTIONS)
    n_queries=2*len(read_json(DATA/'train/annotations.json')['images'])
    estimates={r['profile']:r['seconds_per_query']*n_queries*OPTIONS['epochs']/86400 for r in smoke['profiles'] if r['status']=='passed'}
    print('Rough GPU days per maximum-length trial (smoke timings; excludes full validation):',estimates)
    write_json(run/'timing_estimate.json',{'max_epoch_days_per_trial':estimates,'query_count':n_queries,'caveat':'Synthetic workload timing; real dense masks may be slower.'})
    results=[]
    if RESUME_CHECKPOINT:
        import torch
        trial=torch.load(RESUME_CHECKPOINT,map_location='cpu',weights_only=True)['trial']
        results.append(train_trial(DATA,run/'resumed',trial,PRETRAINED_CHECKPOINT,ALLOW_WEIGHT_DOWNLOAD,RESUME_CHECKPOINT))
    else:
        for profile,lr,seed in itertools.product(profiles,OPTIONS['learning_rates'],OPTIONS['seeds']):
            trial={**OPTIONS,'profile':profile,'lr':lr,'seed':seed}
            trial_name=f'{profile}-lr{lr:g}-seed{seed}'
            results.append(train_trial(DATA,run/trial_name,trial,PRETRAINED_CHECKPOINT,ALLOW_WEIGHT_DOWNLOAD))
            write_json(run/'trials.json',results)
    write_json(run/'trials.json',results)
    winner=max(results,key=lambda r:r['best_AP']);write_json(run/'selection.json',winner)
    print('Validation-selected checkpoint:',winner)
else:
    print('Training disabled. Enable the GPU smoke test first; its measured time informs the training budget.')
